### 콘텐츠 기반 필터링
> 특정 아이템을 기반으로 다른 아이템을 추천

특정영화를 봤을때 감독, 배우, 줄거리, 장르와 같은 영화의 요소들을 기반으로 유사한 영화 추천

In [ ]:
import pandas as pd
movie_df=pd.read_csv('/content/drive/MyDrive/MachineLeaning/영화추천 시스템/tmdb_5000_movies.csv의 사본')
credit_df=pd.read_csv('/content/drive/MyDrive/MachineLeaning/영화추천 시스템/tmdb_5000_credits.csv의 사본')
print(movie_df.head(2))
print(credit_df.head())

In [ ]:
# 두개의 데이터프레임을 하나로 병합. (merge)
df=pd.merge(movie_df,credit_df[['movie_id','cast','crew']],left_on="id",right_on="movie_id",how='left')
df=df.drop('movie_id',axis=1)
df.head(2)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


### 콘텐츠 기반 필터링
적용아이디어 1. 영화줄거리(개요)를 분석해서 비슷한 영화 추천해보기

In [59]:
# 영화 줄거리 정보를 가진 컬럼 확인
df['overview'].head()

,overview
0,"In the 22nd century, a paraplegic Marine is di..."
1,"Captain Barbossa, long believed to be dead, ha..."
2,A cryptic message from Bond’s past sends him o...
3,Following the death of District Attorney Harve...
4,"John Carter is a war-weary, former military ca..."


In [61]:
from numpy import vectorize
# 줄거리(영화개요)에는 관사나 전치사등 불용어가 많음.. 그래서 TF-IDF 사용
# 1)
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(stop_words='english') # 영어단어 기준으로 의미에 영행이 적은 a,am 같은것을 자동제외



In [1]:
# 2) feature를 벡터화 하기전에 'overview' 컬럼안에 결축치(null)이 있으면 벡터화가 안됨.
df['overview'].isnull().values.any() # any()를 쓰면 1개라도 있으면 True
df['overview']=df['overview'].fillna('')
df_null=df[df['overview'] == '']
df_null

NameError: name 'df' is not defined

In [ ]:
# 4) TF-IDF로 줄거리 자연어를 벡터화 하기
# 결과값을 매트릭스 형태 (표-2차원 배열)
tfidf_matrix = vectorizer.fit_transform(df['overview'])
# 매트릭스의 행,열 갯수확인
tfidf_matrix.shape # 4803개의 개요가 있고 총 20978단어가 존재함.
tfidf_matrix # 4083 * 20978칸 중에서 0이 아닌값이 125840개 존재


(4803, 20978)

In [67]:
feature_names= vectorizer.get_feature_names_out()
feature_names[:50]

array(['00', '000', '007', '07am', '10', '100', '1000', '101', '108',
       '10th', '11', '114', '117', '118', '119', '11th', '12', '1200',
       '1215', '1250', '125th', '12th', '13', '1300', '13th', '14', '140',
       '1408', '142', '1429', '148', '14pm', '14th', '15', '150', '150th',
       '1520s', '1536', '15th', '16', '1600s', '161', '1630s', '1644',
       '1681', '1691', '16th', '17', '170', '1700s'], dtype=object)

In [ ]:
# 앞 5개 영화의 TF-IDF 벡터값을 확인
pd.DataFrame(tfidf_matrix[:5].toarray(),columns=feature_names).head()
tfidf_matrix.nnz

# 첫번째 영화에서 0이 아닌 컬럼들의 정보를 얻어오기
tfidf_matrix[0].nonzero() # 0이 아닌 컬럼들의 위치번호가 나옴.
# 0번 영화에 실제로 등장한 단어명과 위치번호(불용어 제외)
indices=tfidf_matrix[0].nonzero()[1]
words = [ feature_names[i] for i in indices]
words

,00,000,007,07am,10,100,1000,101,108,10th,...,zuckerberg,zula,zuzu,zyklon,æon,éloigne,émigré,été,única,über
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 0번 영화에 있는 단어들의 TF-IDF 점수들 확인.
scores=tfidf_matrix[0,indices].toarray().flatten()
list(zip(words,scores))

[('22nd', np.float64(0.3055534742040412)),
 ('century', np.float64(0.19886014555883544)),
 ('paraplegic', np.float64(0.34113864626811446)),
 ('marine', np.float64(0.25802679103767756)),
 ('dispatched', np.float64(0.2786343307183244)),
 ('moon', np.float64(0.2715536698060042)),
 ('pandora', np.float64(0.2924861997916292)),
 ('unique', np.float64(0.24152597853522878)),
 ('mission', np.float64(0.17784472095302578)),
 ('torn', np.float64(0.23864791282019554)),
 ('following', np.float64(0.21536095806128416)),
 ('orders', np.float64(0.25370721950251907)),
 ('protecting', np.float64(0.2655670563059124)),
 ('alien', np.float64(0.20840742038787768)),
 ('civilization', np.float64(0.27493285363270825))]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)
cosine_sim

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.02160533, 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.01488159, 0.        ,
        0.        ],
       ...,
       [0.        , 0.02160533, 0.01488159, ..., 1.        , 0.01609091,
        0.00701914],
       [0.        , 0.        , 0.        , ..., 0.01609091, 1.        ,
        0.01171696],
       [0.        , 0.        , 0.        , ..., 0.00701914, 0.01171696,
        1.        ]])

In [74]:
# 사용자는 영화제목을 입력하면 cosin_sim 매트릭스에서 해당영화의 유사도 높은 영화들을 리턴해줘야 함.
# 영화 제목을 넣으면 영화순번 번호(인덱스)를 주는 기능이 필요함.
# 그래서 pandas에 Series타입으로 이 기능을 구현
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices


,0
title,
Avatar,0
Pirates of the Caribbean: At World's End,1
Spectre,2
The Dark Knight Rises,3
John Carter,4
...,...
El Mariachi,4798
Newlyweds,4799
"Signed, Sealed, Delivered",4800


In [ ]:
indices['Avatar']
df.iloc[4]
cosine_sim[4]
from math import cos
from re import T
# 영화 추천앱에 사용할 기능함수 설계
# 영화의 제목을 파라미터로 받아 코사인 유사도를 통해서 가장 유사도 높은 상위 10갸의 영화목록을 리턴하는 기능함수 만들기
def get_recommendation_movies(title, cosine_sim=cosine_sim):
  # 1.  전달 받은 영화제목의 순번번호 (인덱스번호) 얻어오기
  idx=indices[title]
  # 2. 영화 인덱스 번호와 유사도값을 튜플로 묶은 리스트 만들기
  cosine_sim_scores=list(enumerate(cosine_sim[idx]))
  # 3) 상위 10개를 찾기위해 내림차순 정렬 sorted() 함수
  cosine_sim_scores=sorted(cosine_sim_scores,key = lambda x : x[1],reverse=True)
  # 4) 본인 유사도 제외
  cosine_sim_scores=cosine_sim_scores[1:11]
  # 5) 추출된 10개의 영화 유사도 값에서 영화 인덱스 번호만 추출
  movie_indices=[i[0] for i in cosine_sim_scores]
  # 6) 인덱스값에 해당하는 영화의 제목들을 추출하여 리턴
  return df['title'].iloc[movie_indices]
get_recommendation_movies('The Avengers')

np.int64(0)

### (적용 아이디어 2.) 영화의 다양한 요소기반 추천 (유사도: 장르, 감독, 키워드)

In [2]:
# 영화의 다양한 요소들 (컬럼: genres, cast, crew, keywords)
df.head(2)

NameError: name 'df' is not defined

In [81]:
# 0번 영화의 장르정보 확인
df.loc[0,'genres']

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [82]:
type(df.loc[0,'genres']) # 이거 문자열이어서 딕셔너리형태로 검색이 불가능

str

In [83]:
# 문자열을 리스트로 모듈을 이용해서 해석
from ast import literal_eval
# 리스트 형식을 가진 문자열을 진짜 리스트객체로 변환
features=['cast','crew','genres','keywords']
for feature in features:
  df[feature]=df[feature].apply(literal_eval)



In [84]:
# 변환되었는지 확인
df.loc[0,'genres']

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [85]:
# 1) 컨텐츠 기반 필터링을 위해 각 정보들을 벡터화 하여 코사인 유사도를 계산.
# 각 정보들(감독, 배우, 장르, 키워드)에서 단어를 추출하여 하나의 문자열로 만드는 전처리 작업

In [86]:
# 1) crew 컬럼 데이터에서 '감독' 이름을 뽑아서 'dierector' 라는 새로운 컬럼에 저장
# 감독 이름만 뽑아오는 기능함수
import numpy as np
def get_dierector(crew_list):
  for crew in crew_list:
    if crew['job'] == 'Director':
      return crew['name']
  return np.nan
# 데이터프레임에 '감독' 정보만 있는 컬럼을 새로 생설. 위 함수 적용
df['director']=df['crew'].apply(get_dierector)
df['director']

,director
0,James Cameron
1,Gore Verbinski
2,Sam Mendes
3,Christopher Nolan
4,Andrew Stanton
...,...
4798,Robert Rodriguez
4799,Edward Burns
4800,Scott Smith
4801,Daniel Hsia


In [87]:
# 혹시 감독정보가 비어 있는지 확인
df['director'].isnull().sum() # 결측치는 벡터화가 안됨. 이건 나중에 빈 문자열로 처리

np.int64(30)

In [88]:
# 2) 배우가 너무 많아서 3명만 사용(주연배우들)
# 장르도 확인, 키워드도 너무 많아서 3개만
# 위 3개의 정보 모두 'name'키값에 원하는 정보가 있음.
# 그래서 3개의 컬럼에서 'name' 키값들만 추출하는 함수 만들기
def get_name_list(feature):
  # 각 컬럼들이 list인가?
  if isinstance(feature,list):
    names=[i['name'] for i in feature]
    # 3개 이상이면 앞 3개만 사용
    if len(names) > 3:
      names=names[0:3]

    return names
  return [] #빈 리스트 리턴

In [89]:
# 배우 cast 컬럼의 값을 위 함수에 적용하여 이름만 가진 리스트로 변환
df['cast']= df['cast'].apply(get_name_list)

In [90]:
# 장르  컬럼의 값을 위 함수에 적용하여 이름만 가진 리스트로 변환
df['genres']= df['genres'].apply(get_name_list)

In [91]:
# keywords 컬럼의 값을 위 함수에 적용하여 이름만 가진 리스트로 변환
df['keywords']= df['keywords'].apply(get_name_list)

In [96]:
df[['director','cast','genres','keywords']]

,director,cast,genres,keywords
0,James Cameron,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[Action, Adventure, Fantasy]","[culture clash, future, space war]"
1,Gore Verbinski,"[Johnny Depp, Orlando Bloom, Keira Knightley]","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island]"
2,Sam Mendes,"[Daniel Craig, Christoph Waltz, Léa Seydoux]","[Action, Adventure, Crime]","[spy, based on novel, secret agent]"
3,Christopher Nolan,"[Christian Bale, Michael Caine, Gary Oldman]","[Action, Crime, Drama]","[dc comics, crime fighter, terrorist]"
4,Andrew Stanton,"[Taylor Kitsch, Lynn Collins, Samantha Morton]","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion]"
...,...,...,...,...
4798,Robert Rodriguez,"[Carlos Gallardo, Jaime de Hoyos, Peter Marqua...","[Action, Crime, Thriller]","[united states–mexico barrier, legs, arms]"
4799,Edward Burns,"[Edward Burns, Kerry Bishé, Marsha Dietlein]","[Comedy, Romance]",[]
4800,Scott Smith,"[Eric Mabius, Kristin Booth, Crystal Lowe]","[Comedy, Drama, Romance]","[date, love at first sight, narration]"
4801,Daniel Hsia,"[Daniel Henney, Eliza Coupe, Bill Paxton]",[],[]


In [101]:
# 띄어쓰기와 소문자 변환을 해주는 기능함수 만들기
def clean_data(x):
  # 리스트일 경우
  if isinstance(x,list):
    return [s.replace(' ','').lower() for s in x]
  # 문자열일 경우
  elif isinstance(x,str):
    return x.replace(' ','').lower()
  else:
    return ''


In [122]:
features=['director','cast','genres','keywords']
for feature in features:
  df[feature]=df[feature].apply(clean_data)

df[['director','cast','genres','keywords']]


,director,cast,genres,keywords
0,jamescameron,"[samworthington, zoesaldana, sigourneyweaver]","[action, adventure, fantasy]","[cultureclash, future, spacewar]"
1,goreverbinski,"[johnnydepp, orlandobloom, keiraknightley]","[adventure, fantasy, action]","[ocean, drugabuse, exoticisland]"
2,sammendes,"[danielcraig, christophwaltz, léaseydoux]","[action, adventure, crime]","[spy, basedonnovel, secretagent]"
3,christophernolan,"[christianbale, michaelcaine, garyoldman]","[action, crime, drama]","[dccomics, crimefighter, terrorist]"
4,andrewstanton,"[taylorkitsch, lynncollins, samanthamorton]","[action, adventure, sciencefiction]","[basedonnovel, mars, medallion]"
5,samraimi,"[tobeymaguire, kirstendunst, jamesfranco]","[fantasy, action, adventure]","[dualidentity, amnesia, sandstorm]"
6,byronhoward,"[zacharylevi, mandymoore, donnamurphy]","[animation, family]","[hostage, magic, horse]"
7,josswhedon,"[robertdowneyjr., chrishemsworth, markruffalo]","[action, adventure, sciencefiction]","[marvelcomic, sequel, superhero]"
8,davidyates,"[danielradcliffe, rupertgrint, emmawatson]","[adventure, fantasy, family]","[witch, magic, broom]"
9,zacksnyder,"[benaffleck, henrycavill, galgadot]","[action, adventure, fantasy]","[dccomics, vigilante, superhero]"


In [106]:
# 4) 감독, 배우, 장르, 키워드, 문자열을 줄거리의 단어처럼 하나로 합치기
def create_metadata(x):
  return x['director']+ ' ' + ' '.join(x['cast'])+ ' '.join(x['genres'])+ ' '.join(x['keywords'])
df['metadata'] = df.apply(create_metadata,axis=1)
df['metadata']

,metadata
0,jamescameron samworthington zoesaldana sigourn...
1,goreverbinski johnnydepp orlandobloom keirakni...
2,sammendes danielcraig christophwaltz léaseydou...
3,christophernolan christianbale michaelcaine ga...
4,andrewstanton taylorkitsch lynncollins samanth...
...,...
4798,robertrodriguez carlosgallardo jaimedehoyos pe...
4799,edwardburns edwardburns kerrybishé marshadietl...
4800,scottsmith ericmabius kristinbooth crystallowe...
4801,danielhsia danielhenney elizacoupe billpaxton


In [111]:
# metadata에 있는 문자열을 자연어처리 하려면 벡터화 해야 함.
# 이전 줄거리때처럼 관사(is, a , ..)가 없고 영화의 요소 단어들임.
# 그래서 TF-IDF 방식보다는 빈도수로 벡터화를 수행하는 Bow방식이 적합해보임
# Bow 알고리즘을 구현할 클래스 사용
from sklearn.feature_extraction.text import CountVectorizer
vectorizer=CountVectorizer(stop_words='english')

# 벡터화 실핼
count_metrics=vectorizer.fit_transform(df['metadata'])
count_metrics

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 33900 stored elements and shape (4803, 16146)>

In [113]:
# 6) 벡터화된 코사인유사도 계산
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim2=cosine_similarity(count_metrics,count_metrics)
cosine_sim2

array([[1.   , 0.   , 0.125, ..., 0.   , 0.   , 0.   ],
       [0.   , 1.   , 0.   , ..., 0.   , 0.   , 0.   ],
       [0.125, 0.   , 1.   , ..., 0.   , 0.   , 0.   ],
       ...,
       [0.   , 0.   , 0.   , ..., 1.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , ..., 0.   , 1.   , 0.   ],
       [0.   , 0.   , 0.   , ..., 0.   , 0.   , 1.   ]])

In [117]:
# 7) 위에서 만들었던 함수 재사용
# 영화 이름과 코사인 유사도정보를 주면 해당 영화와 해당 영화와 유사한 영화 10개를 추출해주는 함수 사용
get_recommendation_movies('The Avengers',cosine_sim=cosine_sim2)

,title
26,Captain America: Civil War
79,Iron Man 2
174,The Incredible Hulk
4129,London
7,Avengers: Age of Ultron
33,X-Men: The Last Stand
203,X2
205,Sherlock Holmes: A Game of Shadows
486,The Last Witch Hunter
511,X-Men


In [119]:
get_recommendation_movies('The Avengers',cosine_sim=cosine_sim)

,title
7,Avengers: Age of Ultron
3144,Plastic
1715,Timecop
4124,This Thing of Ours
3311,Thank You for Smoking
3033,The Corruptor
588,Wall Street: Money Never Sleeps
2136,Team America: World Police
1468,The Fountain
1286,Snowpiercer


In [130]:
get_recommendation_movies('Spider-Man 3',cosine_sim=cosine_sim2)

,title
30,Spider-Man 2
159,Spider-Man
659,The Long Kiss Goodnight
4401,The Helix... Loaded
4638,Amidst the Devil's Wings
981,Man of the House
2933,F.I.S.T.
3976,Close Range
4161,The Marine 4: Moving Target
71,The Mummy: Tomb of the Dragon Emperor


### 지금까지 만든 추천 시스템 모델과 유사도값등을 실제 웹앱에 적용하기 위해 별도의 파일(피클)로 저장아여 내보내기

In [132]:
# 객체를 파일로 저장할 수 있는 표준 모듈 pickle
import pickle

# 추천 알고리즘에 사용할 코사인유사도 2개를 파일로 만들기
pickle.dump(cosine_sim,open ('./cosine_sim.pkl','wb'))
pickle.dump(cosine_sim,open ('./cosine_sim2.pkl','wb'))

In [136]:
# 웹 앱 만들때 영화제목과 영화고유번호를 쉽게 연결하기 위해
# 새로운 데이터 프레임 만들기
movies=df[['id','title']]
movies.head()

,id,title
0,19995,Avatar
1,285,Pirates of the Caribbean: At World's End
2,206647,Spectre
3,49026,The Dark Knight Rises
4,49529,John Carter


In [137]:
# 위 데이터프레임도 파일로 저장
pickle.dump(movies,open('./movies.pkl','wb'))

### 실제 웹앱 개발 작업은 코랩환경이 아니라 VScode에서 작업 위에서 저장한 pkl파일들 사용